In [1]:
import requests

url = "https://huggingface.co/datasets/zhengyun21/PMC-Patients-MetaData/resolve/main/PMID2Mesh.json"
file_name = "/Users/ahmadrezaie/2_My_papers/Synthetic_Data_Gen/code/PMC_Patient/PMID2Mesh.json"

# Send a GET request to download the file
response = requests.get(url)
if response.status_code == 200:
    # Write the content to a file
    with open(file_name, "wb") as file:
        file.write(response.content)
    print(f"Downloaded {file_name} successfully!")
else:
    print(f"Failed to download file. Status code: {response.status_code}")

Downloaded /Users/ahmadrezaie/2_My_papers/Synthetic_Data_Gen/code/PMC_Patient/PMID2Mesh.json successfully!


In [50]:
import pandas as pd
import json


# Path to the downloaded JSON file
file_path = "/Users/ahmadrezaie/2_My_papers/Synthetic_Data_Gen/code/PMC_Patient/PMID2Mesh.json"


# Load the JSON data
with open(file_path, "r") as file:
    data = json.load(file)

# Flatten the JSON structure
flattened_data = [
    {"ID": key, "Value": value} for key, values in data.items() for value in values
]

# Convert to DataFrame
df = pd.DataFrame(flattened_data)

# Display the DataFrame
len(df)

# Save to CSV if needed
#df.to_csv("/Users/ahmadrezaie/2_My_papers/Synthetic_Data_Gen/code/PMC_Patient/flattened_PMID2Mesh.csv", index=False)


355662

## unique number of MEsh terms

In [6]:
import json
import pandas as pd
from collections import Counter

# Path to your JSON file
file_path = "/Users/ahmadrezaie/2_My_papers/Synthetic_Data_Gen/code/PMC_Patient/PMID2Mesh.json"

# Load the JSON data
with open(file_path, "r") as file:
    data = json.load(file)

# Flatten the JSON structure into a single list of strings
all_strings = [value for values in data.values() for value in values]

# Count the frequency of each string
string_counts = Counter(all_strings)

# Convert to a DataFrame for easier handling
freq_df = pd.DataFrame(string_counts.items(), columns=["String", "Frequency"])

# Sort by frequency in descending order
freq_df = freq_df.sort_values(by="Frequency", ascending=False)

# Display the results
print("Total unique strings:", len(freq_df))
print(freq_df.head())

# Save to CSV if needed
freq_df.to_csv("""/Users/ahmadrezaie/2_My_papers/Synthetic_Data_Gen/code/PMC_Patient/
string_frequency_distribution.csv""", index=False)


Total unique strings: 13471
         String  Frequency
3        Humans      32277
5          Male      17995
52       Female      16485
49  Middle Aged      10663
0         Adult       9791


In [7]:
freq_df.head()

,String,Frequency
3,Humans,32277
5,Male,17995
52,Female,16485
49,Middle Aged,10663
0,Adult,9791


In [8]:
len(freq_df)

13471

## Mesh

In [10]:
import requests

# URL of the MeSH file
url = "https://nlmpubs.nlm.nih.gov/projects/mesh/2023/xmlmesh/desc2023.xml"
file_name = "/Users/ahmadrezaie/2_My_papers/Synthetic_Data_Gen/code/PMC_Patient/Mesh_desc2023.xml"  # Name of the downloaded file

# Send a GET request to download the file
response = requests.get(url, stream=True)

# Check if the request was successful
if response.status_code == 200:
    with open(file_name, "wb") as file:
        for chunk in response.iter_content(chunk_size=8192):  # Download in chunks
            file.write(chunk)
    print(f"File downloaded successfully: {file_name}")
else:
    print(f"Failed to download file. Status code: {response.status_code}")


File downloaded successfully: /Users/ahmadrezaie/2_My_papers/Synthetic_Data_Gen/code/PMC_Patient/Mesh_desc2023.xml


In [11]:
import xml.etree.ElementTree as ET
from tqdm import tqdm
import pandas as pd

# Path to the downloaded XML file
file_path = "/Users/ahmadrezaie/2_My_papers/Synthetic_Data_Gen/code/PMC_Patient/Mesh_desc2023.xml" 

# Parse the XML file
tree = ET.parse(file_path)
root = tree.getroot()

# Extract relevant data: DescriptorName and TreeNumber
data = []
descriptors = root.findall("DescriptorRecord")

# Wrap the iteration in tqdm for progress tracking
for descriptor in tqdm(descriptors, desc="Processing Descriptors"):
    descriptor_name = descriptor.find(".//DescriptorName/String").text  # Extract the Descriptor name
    tree_numbers = [tn.text for tn in descriptor.findall(".//TreeNumber")]  # Extract all TreeNumbers

    # Append as separate rows for each TreeNumber
    for tree_number in tree_numbers:
        data.append({"DescriptorName": descriptor_name, "TreeNumber": tree_number})

# Convert to a pandas DataFrame
df = pd.DataFrame(data)

# Display the first few rows
print(df.head())

# Save to a CSV if needed
output_path = "/Users/ahmadrezaie/2_My_papers/Synthetic_Data_Gen/code/PMC_Patient/mesh_descriptors.csv"
df.to_csv(output_path, index=False)


Processing Descriptors: 100%|██████████| 30454/30454 [00:00<00:00, 33558.65it/s]


  DescriptorName               TreeNumber
0     Calcimycin      D03.633.100.221.173
1        Temefos      D02.705.400.625.800
2        Temefos      D02.705.539.345.800
3        Temefos      D02.886.300.692.800
4      Abattoirs  J01.576.423.200.700.100


In [12]:
len(df)

63056

## Extracting only disease terms from PMC Patient

In [18]:
pmc= pd.read_csv("""/Users/ahmadrezaie/2_My_papers/Synthetic_Data_Gen/code/PMC_Patient/
string_frequency_distribution.csv""")

mesh= pd.read_csv("/Users/ahmadrezaie/2_My_papers/Synthetic_Data_Gen/code/PMC_Patient/mesh_descriptors.csv")

In [19]:
pmc.head()

,String,Frequency
0,Humans,32277
1,Male,17995
2,Female,16485
3,Middle Aged,10663
4,Adult,9791


In [20]:
mesh.head()

,DescriptorName,TreeNumber
0,Calcimycin,D03.633.100.221.173
1,Temefos,D02.705.400.625.800
2,Temefos,D02.705.539.345.800
3,Temefos,D02.886.300.692.800
4,Abattoirs,J01.576.423.200.700.100


In [36]:
# extracting strings for class C from Mesh:

disease_mesh = mesh[mesh["TreeNumber"].str.startswith("C")]
disease_mesh.head()

,DescriptorName,TreeNumber
8,"Abdomen, Acute",C23.888.592.612.054.200
9,"Abdomen, Acute",C23.888.821.030.249
10,Abdominal Injuries,C26.017
11,Abdominal Neoplasms,C04.588.033
16,Abetalipoproteinemia,C16.320.565.398.500.440.500


In [37]:
len(disease_mesh)

12950

In [39]:
# there seem to be duplicates in names, but with different TreeNumber. I checked Mesh website and it seems, for example, Abdomen, Acute can be reached from different paths. 
# Website: https://www.ncbi.nlm.nih.gov/mesh/?term=Abdomen%2C+Acute
# I am using set to get the unique ones

mesh_disease_list= list(set(disease_mesh["DescriptorName"]))
mesh_disease_list

['Kallmann Syndrome',
 'Pasteurellaceae Infections',
 'Root Resorption',
 'Bruxism',
 'Coma, Post-Head Injury',
 'Sweating, Gustatory',
 'Postcholecystectomy Syndrome',
 'Craniofacial Fibrous Dysplasia',
 'Paresthesia',
 'Dentigerous Cyst',
 'Voice Disorders',
 'Multiple Acyl Coenzyme A Dehydrogenase Deficiency',
 'Chronic Urticaria',
 'Epidermolysis Bullosa Dystrophica',
 'Diabetes Mellitus, Type 1',
 'Costello Syndrome',
 'Tooth, Supernumerary',
 'Hypertrophy, Right Ventricular',
 'Muscle Rigidity',
 'Shaken Baby Syndrome',
 'Glucagonoma',
 'Helminthiasis, Animal',
 'Alcoholism',
 'Panuveitis',
 'Thyroglossal Cyst',
 'Swine Erysipelas',
 'Neoplasms, Neuroepithelial',
 'Signs and Symptoms',
 'Dental Fistula',
 'Renal Tubular Transport, Inborn Errors',
 'Aniseikonia',
 'Thanatophoric Dysplasia',
 'Pneumatosis Cystoides Intestinalis',
 'Tuberculosis, Bovine',
 'Diastema',
 'Chancre',
 'Cat-Scratch Disease',
 'Chondromatosis, Synovial',
 'Pemphigoid, Benign Mucous Membrane',
 'Hereditary

In [40]:
len(mesh_disease_list)

5004

In [41]:
# exctraing disease terms from PMC Patient:

pmc_filtered= pmc[pmc["String"].isin(mesh_disease_list)]
len(pmc_filtered)


4028

In [42]:
pmc_filtered.head(20)

,String,Frequency
17,Lung Neoplasms,1277
25,Postoperative Complications,888
27,"Neoplasm Recurrence, Local",858
29,COVID-19,751
32,Adenocarcinoma,728
35,Liver Neoplasms,689
44,Recurrence,578
46,Skin Neoplasms,557
47,Acute Disease,532
50,Breast Neoplasms,511


In [43]:
pmc["Frequency"].sum()

355662

In [49]:
pmc_filtered["percentage_total"]= (pmc["Frequency"]/pmc["Frequency"].sum())*100
pmc_filtered

/var/folders/wy/vcwn319j4kj7h2gnjqhn60tm0000gn/T/ipykernel_11527/3641551692.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pmc_filtered["percentage_total"]= (pmc["Frequency"]/pmc["Frequency"].sum())*100


,String,Frequency,percentage_total
17,Lung Neoplasms,1277,0.359049
25,Postoperative Complications,888,0.249675
27,"Neoplasm Recurrence, Local",858,0.241240
29,COVID-19,751,0.211156
32,Adenocarcinoma,728,0.204689
...,...,...,...
13443,Oligomenorrhea,1,0.000281
13448,Focal Infection,1,0.000281
13449,Smoldering Multiple Myeloma,1,0.000281
13455,Whiplash Injuries,1,0.000281


In [48]:
= pmc_filtered[pmc_filtered["Frequency"]<=10]

len(ones)

2275